# 01 — Descriptors, Properties, and Class Creation

In [ ]:
# Setup: run this first. On Google Colab (or anywhere the course files are missing) it
# downloads the course and moves into this notebook's folder. Elsewhere it does nothing.
import os, sys
if not os.path.exists("../README.md"):
    course = os.path.join(os.path.expanduser("~"), "python-notebook")
    if not os.path.isdir(course):
        os.system(f"git clone -q --depth 1 https://github.com/SwarmLadder/python-notebook {course}")
    os.chdir(os.path.join(course, "advanced", "notebooks"))
    sys.path.insert(0, os.getcwd())

In [ ]:
from lesson import Exercises, ___, run

check = Exercises('advanced/01', {1: '9365cf553bf2df69', 2: '97eff75bda1510f0', 3: 'c7b8dcdf3d3f8ba2', 4: '8162181c372af23c', 5: 'f5d47e0dc903f580', 6: 'abf18cf81ce559ac', 7: '4312ce334dd33c13', 8: '9d8a90c04db22a6d', 9: 'b6703353a55f290d'}, practice='advanced_01_internals_typing_architecture')

## 1. Properties

- `@property` makes a method read like an attribute; add `@x.setter` to validate assignments.
- Store the real value in a private attribute like `self._price`.
- A property without a setter is read-only: assigning raises `AttributeError`.

In [ ]:
class Item:
    def __init__(self, price):
        self.price = price          # goes through the setter
    @property
    def price(self):
        return self._price
    @price.setter
    def price(self, value):
        if value < 0:
            raise ValueError("price cannot be negative")
        self._price = value

item = Item(5)
item.price = 7
print(item.price)
try:
    item.price = -1
except ValueError as error:
    print(error)

**Exercise 1.** A read-only `area` property. What does assigning to it raise? Write the class name.

In [ ]:
class Sq:
    def __init__(self, s): self.s = s
    @property
    def area(self): return self.s ** 2

try:
    Sq(2).area = 9
except Exception as error:
    raised = type(error).__name__
predicted = ___
check(1, predicted)

**Exercise 2.** Reject negative prices in the setter.

In [ ]:
class Item:
    @property
    def price(self): return self._price
    @price.setter
    def price(self, value):
        if ___:
            raise ValueError("negative")
        self._price = value

item = Item()
item.price = 3
try:
    item.price = -2
    outcome = "accepted"
except ValueError:
    outcome = "rejected"
check(2, (item.price, outcome))

## 2. Descriptors

- A descriptor is a class attribute object with `__get__` / `__set__`. `property` is one; you can write your own to reuse validation.
- `__set_name__(self, owner, name)` tells it which attribute name it was given.
- Store each instance's value in `instance.__dict__[self.name]`. `__get__` receives `instance=None` when accessed on the class: return `self` then.

In [ ]:
class PositiveInt:
    def __set_name__(self, owner, name):
        self.name = name
    def __get__(self, instance, owner=None):
        if instance is None:
            return self
        return instance.__dict__[self.name]
    def __set__(self, instance, value):
        if not isinstance(value, int):
            raise TypeError(f"{self.name} must be an int")
        if value <= 0:
            raise ValueError(f"{self.name} must be positive")
        instance.__dict__[self.name] = value

class Order:
    quantity = PositiveInt()
    def __init__(self, quantity):
        self.quantity = quantity

print(Order(3).quantity, type(Order.quantity).__name__)
try:
    Order(0)
except ValueError as error:
    print(error)

**Exercise 3.** With `quantity = PositiveInt()`, what name does `__set_name__` receive?

In [ ]:
predicted = ___
check(3, predicted)

**Exercise 4.** Accessed on the class (`Order.quantity`), `__get__` gets `instance=None`. What should it return? Write `"self"` or `"None"`.

In [ ]:
answer = ___
check(4, answer)

**Exercise 5.** Which checks fail? Predict the error class names for `"3"`, `0` and `5`, using `"ok"` for success.

In [ ]:
predicted = ___   # a list of three strings
check(5, predicted)

**Exercise 6.** Booleans are ints in Python. Is `isinstance(True, int)` true?

In [ ]:
predicted = ___
check(6, predicted)

## 3. Class creation hooks

- `__init_subclass__(cls, **kwargs)` runs on the base class each time a subclass is created. Use it to register or validate subclasses.
- Read attributes defined on the subclass itself with `cls.__dict__.get("name")`, not inherited ones.
- `type(name, bases, namespace)` builds a class dynamically. Metaclasses exist, but `__init_subclass__` covers most needs.

In [ ]:
class Plugin:
    registry = {}
    def __init_subclass__(cls, **kwargs):
        super().__init_subclass__(**kwargs)
        name = cls.__dict__.get("name")
        if not isinstance(name, str):
            raise TypeError(f"{cls.__name__} must define a string `name`")
        Plugin.registry[name] = cls

class Csv(Plugin):
    name = "csv"

print(Plugin.registry)
Point = type("Point", (), {"x": 0})
print(Point.__name__, Point.x)

**Exercise 7.** Register each subclass by its `name`.

In [ ]:
class Base:
    registry = {}
    def __init_subclass__(cls, **kwargs):
        super().__init_subclass__(**kwargs)
        Base.registry[cls.name] = ___

class A(Base):
    name = "a"

check(7, Base.registry["a"].__name__)

**Exercise 8.** Reject a duplicate name: is it already in the registry?

In [ ]:
registry = {"csv": object}
name = "csv"
duplicate = name in ___
check(8, duplicate)

**Exercise 9.** Build a class with `type`. What is `Point.x`?

In [ ]:
Point = type("Point", (), {"x": 5})
predicted = ___
check(9, predicted)

## Challenges

Real functions, graded by tests. Fill every `___`, run the function cell, then run its check.

### 1. `PositiveInt`

A descriptor that only accepts positive ints.

In [ ]:
class PositiveInt:
    """Descriptor that only accepts positive ints (TypeError / ValueError otherwise).

    Usage:  class Order:  quantity = PositiveInt()
    Hint: __set_name__ learns the attribute name; store values in instance.__dict__.
    """

    def __set_name__(self, owner, name: str) -> None:
        self.name = ___

    def __get__(self, instance, owner=None):
        if ___:
            return self
        return instance.__dict__[self.name]

    def __set__(self, instance, value) -> None:
        if ___:
            raise TypeError(f"{self.name} must be an int")
        if ___:
            raise ValueError(f"{self.name} must be positive")
        instance.__dict__[self.name] = value

In [ ]:
check('PositiveInt')

### 2. `Plugin`

Subclasses register themselves by `name`; missing or duplicate names raise `TypeError`.

In [ ]:
class Plugin:
    """Base class whose subclasses register themselves by `name` via __init_subclass__.

    Plugin.registry maps name -> subclass. A subclass without a `name` class attribute,
    or with a duplicate name, raises TypeError at class-creation time.
    """
    registry: dict[str, type] = {}

    def __init_subclass__(cls, **kwargs):
        super().__init_subclass__(**kwargs)
        name = cls.__dict__.get("name")
        if ___:
            raise TypeError(f"{cls.__name__} must define a string `name`")
        if ___:
            raise TypeError(f"duplicate plugin name {name!r}")
        Plugin.registry[name] = cls

In [ ]:
check('Plugin')

## Quick quiz

Replace each `'?'` in the cell below with your choice and run it. The answers are hidden, but each one is checked.

**1.** A @property is implemented as a…

   - **a)** Metaclass
   - **b)** Descriptor
   - **c)** Decorator that copies the method

**2.** __set_name__ is called…

   - **a)** When an instance is deleted
   - **b)** When the owner class is created
   - **c)** On every attribute access

**3.** Before writing a metaclass, prefer…

   - **a)** Global variables
   - **b)** Decorators, __init_subclass__, or plain functions
   - **c)** Two metaclasses

In [ ]:
from quiz import Quiz

quiz = Quiz('advanced/01', {1: 'c6edf5295ed5', 2: '591fe465f1c7', 3: '61662b4b3d92'})
quiz.answer(1, '?')
quiz.answer(2, '?')
quiz.answer(3, '?')

**Next:** [02 — Object References, Copying, Garbage Collection, and Weakrefs](02_memory_references_weakrefs.ipynb)